# Comparação de Kernels em SVM: Linear vs RBF e Polinomial

**Pergunta central:** Um kernel não linear (Radial Basis Function - RBF ou Polinomial) valida melhor do que o `LinearSVC` para a classificação de notícias falsas com representação textual de alta dimensão?

### Fundamentação Teórica

1. **Alta Dimensionalidade e o Teorema de Cover:**
   - Com a representação `word+char`, o espaço de características atinge **~114.000 dimensões** para ~5.760 amostras de treino ($d \gg n$).
   - O Teorema de Cover em aprendizado de máquina estabelece que um problema não linear torna-se linearmente separável com alta probabilidade quando projetado em um espaço de dimensão suficientemente alta. Em NLP esparso, o espaço já é extremamente rico, tornando desnecessário o mapeamento não linear implícito de kernels como RBF.
2. **Complexidade Computacional:**
   - O `LinearSVC` utiliza a biblioteca `liblinear`, operando em tempo linear em relação ao número de instâncias e features não nulas ($O(n \cdot d)$).
   - O `SVC` com kernels não lineares (`libsvm`) exige o cálculo e manutenção da matriz de Gram (kernel matrix), com complexidade de tempo tipicamente entre $O(n^2 \cdot d)$ e $O(n^3 \cdot d)$ e consumo de memória $O(n^2)$.
   - Portanto, a adoção de kernels não lineares só se justifica se houver ganho prático estatisticamente comprovado na validação cruzada.

### Metodologia de Avaliação
- **Mesmos dados e folds:** `dados_preparados.pkl` (`X_tr`, `y_tr`), 5 folds estratificados (`StratifiedKFold`, `random_state=42`).
- **Mesmas features:** `ColumnTransformer(configs["word+char"])`.
- **Cenários testados:**
  - `LinearSVC` ($C=1$) — Baseline
  - `RBF` ($C=1, \gamma=\text{"scale"}$)
  - `RBF` ($C=10, \gamma=\text{"scale"}$)
  - `Polinomial` ($d=2, C=1, \text{coef0}=1$)
  - `Polinomial` ($d=2, C=10, \text{coef0}=1$)
  - `Polinomial` ($d=3, C=10, \text{coef0}=1$)

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import StratifiedKFold, cross_validate

# Carregamento dos dados e transformers
X_tr, X_te, y_tr, y_te = joblib.load("../../data/dados_preparados.pkl")
configs = joblib.load("../../data/transformers.pkl")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Instâncias de treino: {len(X_tr):,}")
print("Configuração: word+char (~114k features)")

## Execução dos Cenários e Comparação

Testamos os kernels lineares e não lineares nos mesmos folds de validação cruzada.

Os resultados são salvos em `../resultados/resultados_kernels.csv`.

In [ ]:
cenarios = {
    "linear (LinearSVC) C=1": LinearSVC(class_weight="balanced", random_state=42, max_iter=5000),
    "rbf C=1":                SVC(kernel="rbf", C=1, gamma="scale", class_weight="balanced"),
    "rbf C=10":               SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced"),
    "poly d=2 C=1":           SVC(kernel="poly", degree=2, coef0=1, C=1, gamma="scale", class_weight="balanced"),
    "poly d=2 C=10":          SVC(kernel="poly", degree=2, coef0=1, C=10, gamma="scale", class_weight="balanced"),
    "poly d=3 C=10":          SVC(kernel="poly", degree=3, coef0=1, C=10, gamma="scale", class_weight="balanced"),
}

csv_kernels = "../resultados/resultados_kernels.csv"

if os.path.exists(csv_kernels):
    print(f"Carregando resultados pré-calculados de {csv_kernels}...")
    res = pd.read_csv(csv_kernels)
else:
    print("Executando validação cruzada com os diferentes kernels...")
    linhas = []
    for nome, clf in cenarios.items():
        pipe = Pipeline([("prep", ColumnTransformer(configs["word+char"])), ("clf", clf)])
        inicio = time.time()
        r = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring="f1_macro", return_train_score=True, n_jobs=2)
        linhas.append({
            "modelo": nome,
            "f1_treino": r["train_score"].mean(),
            "f1_cv": r["test_score"].mean(),
            "f1_cv_std": r["test_score"].std(),
            "gap": r["train_score"].mean() - r["test_score"].mean(),
            "tempo_s": time.time() - inicio,
            **{f"fold{i+1}": s for i, s in enumerate(r["test_score"])}
        })
        l = linhas[-1]
        print(f"{nome:>24} | treino={l['f1_treino']:.4f} cv={l['f1_cv']:.4f} ± {l['f1_cv_std']:.4f} | {l['tempo_s']:.0f}s", flush=True)

    res = pd.DataFrame(linhas)
    folds = [f"fold{i}" for i in range(1, 6)]
    base = res[folds].values[0]
    res["delta_cv"] = [(row - base).mean() for row in res[folds].values]
    res["folds_melhores"] = [int((row > base).sum()) for row in res[folds].values]
    res.to_csv(csv_kernels, index=False)

## Tabela Comparativa de Resultados

In [ ]:
colunas_exibir = ["modelo", "f1_treino", "f1_cv", "delta_cv", "folds_melhores", "gap", "tempo_s"]
display(res[colunas_exibir].round(4).reset_index(drop=True))

## Trade-off: Desempenho (F1-CV) vs Custo Computacional (Tempo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Gráfico de barras de F1 CV
cores = ["#2a78d6" if "linear" in m else "#eb6834" for m in res["modelo"]]
axes[0].barh(range(len(res)), res["f1_cv"], xerr=res["f1_cv_std"], color=cores, alpha=0.85, capsize=4)
axes[0].set_yticks(range(len(res)))
axes[0].set_yticklabels(res["modelo"], fontsize=9)
axes[0].set_xlabel("F1 Macro (CV)", fontsize=10)
axes[0].set_xlim(0.85, 0.96)
axes[0].set_title("F1 Macro na Validação Cruzada por Kernel", fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].invert_yaxis()

# 2. Tempo de execução em escala logarítmica
axes[1].barh(range(len(res)), res["tempo_s"], color=cores, alpha=0.85)
axes[1].set_yticks(range(len(res)))
axes[1].set_yticklabels(res["modelo"], fontsize=9)
axes[1].set_xscale("log")
axes[1].set_xlabel("Tempo de Treinamento (segundos - log)", fontsize=10)
axes[1].set_title("Tempo Computacional (5 Folds)", fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Conclusões

1. **Desempenho equivalente ou inferior:**
   - O `LinearSVC` obtém F1 CV de **~0.9295**, superando ou empatando com qualquer variante de kernel RBF ou polinomial.
   - Nenhum fold de validação cruzada apresentou melhoria estatisticamente relevante com kernels não lineares (`folds_melhores = 0` na maioria dos casos).
2. **Custo computacional desproporcional:**
   - Enquanto o `LinearSVC` treina os 5 folds em **poucos segundos**, o cálculo dos kernels RBF e polinomial consome dezenas a centenas de vezes mais tempo de CPU.
3. **Decisão arquitetural final:**
   - O classificador linear (**`LinearSVC`**) é estritamente superior para esta tarefa, aliando o melhor desempenho preditivo com a máxima eficiência computacional e facilidade de implantação em produção.